# IMDB Reviews - Preprocessing


In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from pathlib import Path
import nltk

In [2]:
# Note: Run this cell only once for the first time to download the necessary NLTK resources
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kaung\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kaung\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\kaung\AppData\Roaming\nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\kaung\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

## 1) Load data

In [36]:
DATA_PATH = Path("../data/IMDB Dataset.csv")
df = pd.read_csv(DATA_PATH)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 2) Cleaning with negation handling and lemmatization

In [38]:
# Regexes
URL_RE   = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
HTML_RE  = re.compile(r'<.*?>')
NUM_RE   = re.compile(r'\d+')

lemmatizer = WordNetLemmatizer()

NEGATORS = {"not","no","never","n't","nor"}
STOPWORDS = ENGLISH_STOP_WORDS
SCOPE_END = re.compile(r'^[\.!?,;:]$')

def apply_negation(tokens):
    out, negate = [], False
    for t in tokens:
        if SCOPE_END.match(t):
            negate = False
            continue
        low = t.lower()
        if low in NEGATORS:
            negate = True
            out.append(low)
            continue
        out.append(low + '_NEG' if negate else low)
    return out

def clean_text(text: str):
    s = str(text).lower()
    s = HTML_RE.sub(' ', s)
    s = URL_RE.sub(' ', s)
    s = NUM_RE.sub(' num ', s)

    tokens = word_tokenize(s)
    tokens = apply_negation(tokens)

    # Remove stopwords among word tokens only 
    processed = [t for t in tokens if t.endswith('_NEG') or (re.fullmatch(r"[A-Za-z-]+", t) and t not in STOPWORDS)]

    lemmas = []
    bases = [t[:-4] if t.endswith('_NEG') else t for t in processed]
    for t, base in zip(processed, bases):
        if re.fullmatch(r"[A-Za-z']+(_NEG)?", t):
            lm = lemmatizer.lemmatize(base)
            t = (lm + '_NEG') if t.endswith('_NEG') else lm
        lemmas.append(t)
    
    return ' '.join(lemmas)

## 3) Apply cleaning

In [41]:
df['review'] = df['review'].map(clean_text)
df.head(5)

,review,sentiment
0,reviewer mentioned watching just num oz episod...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically family little boy jake think zombie ...,negative
4,petter mattei love time money visually stunnin...,positive


## 5) Save cleaned dataset

In [40]:
out_path = Path('../data/IMDB_clean.csv')
df.to_csv(out_path, index=False)
out_path

WindowsPath('../data/IMDB_clean.csv')